# MVP — Machine Learning & Analytics

**Nome:** _Maria Eduarda Alves Esteves_  
**Matrícula:** _RA:4052026000019_  
**Data:** _04/07/2026_  
**Dataset:** _MVP_ML https://github.com/meduardaae/MVP_ML/blob/main/pca_90_components.npz_  
**Tipo de problema:** _Redução de dimensionalidade / Visualização de dados / Aprendizado não supervisionado_  

---

## Descrição

Este notebook realiza uma análise exploratória de dados derivados de simulações de dinâmica molecular de proteínas de coronavírus. Os dados de entrada consistem em coordenadas obtidas por Análise de Componentes Principais (PCA), preservando aproximadamente 90% da variância das matrizes de correlação entre resíduos de aminoácidos.

#1. Definição do problema

É necessário voltar no tempo para explicar o **contexto**: Os dados utilizados neste trabalho foram gerados para o meu projeto de doutorado. O objetivo do estudo é investigar se os padrões de correlação entre resíduos de aminoácidos em um mesmo complexo proteico são conservados ou apresentam diferenças entre espécies distintas da família dos coronavírus.
Inicialmente, foram realizadas simulações de dinâmica molecular para proteínas de três espécies de coronavírus: SARS-CoV, SARS-CoV-2 e MERS-CoV. A partir das trajetórias geradas, foram calculadas matrizes de correlação entre os resíduos de aminoácidos, representando a comunicação dinâmica entre diferentes regiões da proteína ao longo da simulação.
O conjunto de dados original é composto por 20 réplicas independentes de 500 ns para cada sistema viral, totalizando:

>SARS-CoV: 20 matrizes de correlação (249 × 249)

>SARS-CoV-2: 20 matrizes de correlação (249 × 249)

>MERS-CoV: 20 matrizes de correlação (249 × 249)

Em cada matriz, os 249 resíduos de aminoácidos são representados tanto nas linhas quanto nas colunas, enquanto os valores armazenados correspondem aos coeficientes de correlação entre pares de resíduos.
Foi aplicada a técnica de Análise de Componentes Principais (PCA) para reduzir o número de variáveis. Como resultado, foram selecionados os 25 primeiros componentes principais (PCs), suficientes para explicar aproximadamente 90% da variância total do conjunto de dados.

**Agora** os PCs obtidos por PCA constituem a entrada deste notebook. A partir deles, será aplicada a técnica de MDS a fim de projetar os dados em um espaço bidimensional, preservando as relações de distância entre as amostras e permitindo a análise visual da separação entre os sistemas SARS-CoV, SARS-CoV-2 e MERS-CoV.

###1.1 Objetivo

Aplicar a técnica de MDS para comprimir os 25 PCs de matrizes de correlação em um mapa 2D interpretável, a fim de discriminar visualmente os sistemas SARS, SARS2 e MERS.

### 1.2 Tipo de problema



- **Aprendizado não supervisionado:** _redução de dimensionalidade_

**Justificativa:** _Este trabalho utiliza Multidimensional Scaling (MDS), uma técnica de aprendizado não supervisionado empregada para redução de dimensionalidade e visualização de dados. O objetivo não é prever classes, valores numéricos ou observações futuras, mas projetar amostras de alta dimensionalidade em um espaço bidimensional preservando, tanto quanto possível, as relações de distância entre elas._

### 1.3 Hipóteses e critérios de sucesso

- Hipótese 1. As 25 componentes principais selecionadas (que filtram o ruído e retêm 90% da variância das matrizes originais) guardam informações geométricas e topológicas cruciais sobre a comunicação dinâmica dos aminoácidos.

- Hipótese 2. O algoritmo de MDS  consegue comprimir com sucesso esse espaço latente de 25 dimensões para um plano 2D sem gerar distorções severas.
- Hipótese 3. A projeção gerada pelo MDS resulta em um mapa bidimensional mais fiel às distâncias originais do que olhar apenas para o plano bruto simplificado dos PC1 vs PC2.

**Métrica principal**: Coeficiente de Correlação de Pearson entre a matriz de distâncias par a par no espaço original (25D) e a matriz de distâncias obtida no espaço projetado (2D).

**Resultado mínimo esperado**: Obter uma correlação de Pearson superior a 0,80 no MDS, comprovando a eficiência do manifold.

**Restrição prática**: Interpretabilidade visual e estabilidade estatística. O mapa 2D final precisa permitir identificação visual da proximidade ou isolamento entre os clusters das três espécies SARS, SARS2 e MERS através de seus centróides.

A documentação da técnica do MDS está disponível em: https://scikit-learn.org/stable/modules/generated/sklearn.manifold.MDS.html

# 2. Ambiente, bibliotecas e reprodutibilidade

In [ ]:
import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.spatial.distance import pdist
from scipy.stats import pearsonr
from sklearn.manifold import MDS
from io import BytesIO
import requests

### 2.1 Carregamento da fonte dos dados de PCA
Aqui será carregado o arquivo pca_90_components.npz, que contêm os dados dos PCs no formato Numpy. No Github também é possível acessar os dados no formato csv.

O conteúdo é publico e foi selecionado para realizar o aprimoramento na análise dos dados.

In [ ]:
#Carregamento do arquivo npz hospedado no Github
NPZ_URL = "https://github.com/meduardaae/MVP_ML/raw/refs/heads/main/pca_90_components.npz"
SYSTEM_NAMES = {1: "SARS", 2: "SARS2", 3: "MERS"}
COLORS = {1: "#E74C3C", 2: "#3498DB", 3: "#2ECC71"}
MARKERS = {1: "o", 2: "s", 3: "^"}

SEED = 42

### 2.2 Dependências adicionais

In [ ]:
for lib in ["numpy", "pandas", "matplotlib", "sklearn", "scipy", "requests"]:
    __import__(lib)


### 2.3 Funções auxiliares

Aqui são carregadas as funções:

1) `load_npz`: Baixa automaticamente o arquivo npz com os dados do PCA no formato numpy, lê o arquivo e identifica os dados.

2) `run_mds`: Executa o algoritmo do MDS, garantindo o achatamento dos dados para duas dimensões (n_components=2)

3) `dist_correlation`: Calcula a distância euclidiana, entre os pontos do mapa 2D obtido com MDS, e os pontos dos dados originais do input. Em seguida, calcula a correlação de Pearson; que determina valores entre -1 a +1, para avaliar se o mapa 2D dirtorceu ou não, a realidade das distâncias originais.

4) `classify_stress`: Calcula a estatística que interpreta o erro de convergência do algortimo (Stress). Ela é utilizada para avaliar o desempenho entre o modelo métrico e não métrico do MDS.

5) `plot_embedding`: Permite avaliar visualmente os resultados obtidos com as métricas matemáticas.




In [ ]:
def load_npz(url: str):
    #Baixa o arquivo NPZ contendo as coordenadas filtradas pelo PCA e metadados.
    resp = requests.get(url, timeout=30)
    resp.raise_for_status()
    data = np.load(BytesIO(resp.content), allow_pickle=True)
    keys = list(data.keys())

    coords = data["pcs_90_coordinates"] if "pcs_90_coordinates" in keys else data["coordinates90"]
    labels = data["system_labels"] if "system_labels" in keys else data["systemlabels"]

    n_components = int(data["n_components_90"]) if "n_components_90" in keys else coords.shape[1]
    variance_captured = float(data["actual_variance_captured"]) if "actual_variance_captured" in keys else None
    cumulative_var = data["all_cumulative_variance"] if "all_cumulative_variance" in keys else None

    if "system_names" in keys:
        system_names = data["system_names"].item() if data["system_names"].ndim == 0 else {int(i+1): str(n) for i, n in enumerate(data["system_names"])}
    else:
        system_names = {1: "SARS-CoV", 2: "SARS-CoV-2", 3: "MERS-CoV"}

    data.close()
    return coords, labels, n_components, variance_captured, cumulative_var, system_names


def run_mds(X, metric=True, max_iter=300, n_init=4, seed=SEED):


    #metric=True define o MDS Métrico
    #metric=False define o MDS Não-Métrico

    model = MDS(
        n_components=2,
        metric=metric,
        max_iter=max_iter,
        n_init=n_init,              # Escolher o melhor ponto de partida através de testes aleatórios
        random_state=seed,          # Travar o gerador aleatório para manter a reprodutibilidade do mapa
        normalized_stress='auto'    # Ativar o cálculo automático do Stress normalizado (-1 a 1) se for Não-Métrico
    )
    coords = model.fit_transform(X)
    return coords, model.stress_


def dist_correlation(d_orig, X_2d):
    #Mede a fidelidade geométrica calculando a correlação de Pearson das distâncias
    d_2d = pdist(X_2d, metric='euclidean')
    corr, _ = pearsonr(d_orig, d_2d)
    return corr, d_2d


def classify_stress(stress, metric=True):
    #Analisa o desempenho das abordagens métrica e não-métrica.

    if stress is None:
        return "N/A"
    if not metric:
        if stress < 0.05: return "Excelente"
        if stress < 0.10: return "Bom"
        if stress < 0.20: return "Regular"
        return "Ruim"
    return "Stress bruto*"

def plot_embedding(ax, coords, labels, title, show_centroids=True):
    for sid, name in SYSTEM_NAMES.items():
        mask = (labels == sid)
        ax.scatter(
            coords[mask, 0],
            coords[mask, 1],
            color=COLORS[sid],
            marker=MARKERS[sid],
            s=50,
            alpha=0.7,
            label=name,
            edgecolors='w'
        )
        if show_centroids:
            centroid = coords[mask].mean(axis=0)
            ax.scatter(
                centroid[0],
                centroid[1],
                color=COLORS[sid],
                marker='*',  #Identificação visual dos centróides
                s=300,
                edgecolors='k',
                linewidth=1.5,
                zorder=10
            )
    ax.set_title(title)
    ax.set_xlabel("PC1")
    ax.set_ylabel("PC2")
    ax.legend(title="Espécie")
    ax.grid(alpha=0.3)

#3. Execução do carregamento dos dados e identificação das *features*

In [ ]:
coords90, system_labels, n_components, variance_captured, cumulative_var, SYSTEM_NAMES = load_npz(NPZ_URL)
unique_ids, counts = np.unique(system_labels, return_counts=True)


###3.1 Análise exploratória dos dados
Nesta sessão são plotados os gráficos com os dados obtidos no tópico 2.3

In [ ]:
ig, axes = plt.subplots(1, 3, figsize=(16, 4))

#Confirmação da quantidade de dados por espécie
for sid, name in SYSTEM_NAMES.items():
    axes[0].bar(name, (system_labels == sid).sum(), color=COLORS[sid], alpha=0.85)
axes[0].set_title("Distribuição de dados entre espécies")
axes[0].set_ylabel("Quantidade de matrizes")
axes[0].grid(axis="y", alpha=0.3)

#Avaliação da distribuição linear dos dois primeiros PCs
for pc_idx, ax in zip([0, 1], axes[1:]):
    for sid, name in SYSTEM_NAMES.items():
        mask = (system_labels == sid)
        ax.hist(coords90[mask, pc_idx], bins=12, alpha=0.5, color=COLORS[sid], label=name)
    ax.set_title(f"Distribuição de PC{pc_idx + 1}")
    ax.set_xlabel(f"PC{pc_idx + 1}")
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()


#Avaliação de outliers dos PCs que representam a maior variancia do conjunto de dados
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, pc_idx in zip(axes, [0, 1]):
    data_by_system = [coords90[system_labels == sid, pc_idx] for sid in sorted(SYSTEM_NAMES)]
    ax.boxplot(data_by_system, labels=list(SYSTEM_NAMES.values()))
    ax.set_title(f"Boxplot PC{pc_idx + 1}")
    ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

### Síntese da análise exploratória

- **Balanceamento:** Perfeitamente balanceado (20 amostras por sistema viral).
- **Duplicatas:** Nenhuma identificada.
- **Escala:** PC1-PC25 já estruturados pelo PCA.
- **Padrões iniciais:** Há forte sobreposição de réplicas de SARS e SARS-CoV-2 quando olhamos apenas os eixos lineares básicos do PCA.
- **Outliers:** Distribuição contínua sem anomalias extremas.


#4. Preparação dos dados para redução da dimensionalidade e *double check* de integridade
- Isolamento das variáveis x (coordenadas dos componentes) e y (rótulo por espécie)
- Teste de valores nulos
- Avaliação de duplicata

In [ ]:
X = coords90.copy()
y = system_labels.copy()

assert np.isnan(X).sum() == 0, "Erro: Valores ausentes detectados."
assert pd.DataFrame(X).duplicated().sum() == 0, "Erro: Linhas duplicadas detectadas."

# Aqui a distância euclidiana é de fato calculada
d_orig = pdist(X, metric='euclidean')
std_per_pc = X.std(axis=0)


#6. Execução do MDS, compressão dimensional e referência visual

- O teste de referência isola os dois PCs e calcula a fidelidade entre eles em relação ao espaço das demais dimensões. A ideia é entender o que acontece se P3-PC25 forem ignorados.
- Em seguida é calculado o MDS Não-Métrico com as 25 dimensões.
- Comparativamente, é calculado o MDS Métrico com as mesma dimensões.

In [ ]:
# Referência restrita: Isolamento de PC1 e PC2
coords_baseline = X[:, :2]
corr_baseline, _ = dist_correlation(d_orig, coords_baseline)

# MDS Não-métrico
coords_nonmetric, stress_nonmetric = run_mds(X, metric=False)
corr_nonmetric, _ = dist_correlation(d_orig, coords_nonmetric)

# MDS métrico
coords_metric, stress_metric = run_mds(X, metric=True)
corr_metric, _ = dist_correlation(d_orig, coords_metric)


###6.1 Avaliação dos modelos Métrico e Não Métrico na redução das 25 dimensões

In [ ]:
# Cálculo do ganho ou perda de fidelidade em relação à referência restrita
ganho_mds_metrico = corr_metric - corr_baseline
ganho_mds_nao_metrico = corr_nonmetric - corr_baseline

print(f"1. Referência Linear (PCA 2D)  : Fidelidade (r) = {corr_baseline:.4f}")
print(f"2. MDS Não-Métrico   : Fidelidade (r) = {corr_nonmetric:.4f} | Ganho vs Ref: {ganho_mds_nao_metrico:+.4f} | Stress: {stress_nonmetric:.4f} ({classify_stress(stress_nonmetric, metric=False)})")
print(f"3. MDS Métrico: Fidelidade (r) = {corr_metric:.4f} | Ganho vs Ref: {ganho_mds_metrico:+.4f} | Stress Bruto: {stress_metric:.2f}")


Através das métricas de Stress e do ganho de fidelidade calculadas, foi selecionado o MDS Métrico para seguir com a redução dos dados.

###6.2 Ajuste fino dos hiperparâmetros do MDS Métrico
- Configuração do espaço de busca do algoritmo (3 valores de max_iter x 3 valores de n_init)
- Organização do loop `itertools.product`: a cada 9 combinações do grid atual, armazena a informação e calcula as métricas
- Seleção do grid que obteve a maior correlação de Pearson
- Impressão do resultado dos cálculos. O melhor grid será destacado com uma estrela

In [ ]:
#Ajuste do grid search
param_grid = {
    "metric": [True],
    "max_iter": [300, 500, 1000],    # Teste máximo de iteração para convergência
    "n_init": [4, 8, 16],            # Teste de múltiplas inicializações contra mínimos locais
}

#Loop
search_results = []
for metric, max_iter, n_init in itertools.product(param_grid["metric"], param_grid["max_iter"], param_grid["n_init"]):
    # Executa o MDS para a combinação atual do grid search
    coords, stress = run_mds(X, metric=metric, max_iter=max_iter, n_init=n_init, seed=SEED)
    corr, _ = dist_correlation(d_orig, coords)

    # Armazena os resultados para avaliação posterior
    search_results.append({
        "metric": metric, "max_iter": max_iter, "n_init": n_init,
        "stress": stress, "corr": corr, "coords": coords
    })

#Seleção do melhor grid
best = max(search_results, key=lambda r: r["corr"])
coords_best = best["coords"]


#Print do resultado
df_search = pd.DataFrame([{
    "Max Iter": r["max_iter"],
    "N Init": r["n_init"],
    "Stress Bruto": f"{r['stress']:.2f}",
    "Corr. Distâncias": f"{r['corr']:.4f}",
    "Melhor": "★" if r is best else ""
} for r in search_results])

print(df_search.to_string(index=False))

###6.3 Diagnóstico de fidelidade do melhor grid
- Calculo do erro absoluto entre o mapa 2D e o espaço original 25D
- Diagrama de Shepard: cruza cada distância real 25D (X) com sua respectiva distância projetada 2D (Y). A linha tracejada vermelha representa a perfeição matemática (y=x)

In [ ]:
_, dist_best = dist_correlation(d_orig, coords_best)
errors = np.abs(d_orig - dist_best)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(errors, bins=25, color="steelblue", alpha=0.85, edgecolor="black")
axes[0].axvline(errors.mean(), color="red", linestyle="--", label=f"Média: {errors.mean():.4f}")
axes[0].axvline(np.median(errors), color="orange", linestyle="--", label=f"Mediana: {np.median(errors):.4f}")
axes[0].set_title("Distribuição do Erro Absoluto")
axes[0].set_xlabel("Erro (Distância Real − Distância Projetada)")
axes[0].legend()
axes[0].grid(alpha=0.3)

# Geração do Diagrama de Shepard para atestar a qualidade da compressão 25D -> 2D
axes[1].scatter(d_orig, dist_best, alpha=0.4, s=12, color="steelblue")
min_d, max_d = d_orig.min(), d_orig.max()
axes[1].plot([min_d, max_d], [min_d, max_d], "r--", alpha=0.7, label="Preservação Ideal (y=x)")
axes[1].set_title(f"Diagrama de Shepard\nCorrelação Conquistada r = {best['corr']:.4f}")
axes[1].set_xlabel("Distâncias no Espaço Filtrado (25D)")
axes[1].set_ylabel("Distâncias no Plano Compactado (2D)")
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()


- Análise dos resultados:

O grafico do **erro absoluto** avalia a distorção da distância. Sabendo que o input inicial tem 60 amostras (20 para cada espécie), a combinação aleatória entre cada um dos pares resulta em 1.770 pares de distâncias.

Assim, o eixo y representa a frequência absoluta desses pares, e, x é a projeção do erro. Uma vez que o gráfico apresenta assimetria positiva (a frequência diminui a medida que o erro aumenta), indica que a maior parte dos pares de distância sofreu baixa distorção.

Já o gráfico do **Diagrama de Shepard** cruza cada distância real 25D (eixo X) com sua respectiva distância projetada 2D (eixo Y). A linha tracejada vermelha representa a perfeição matemática (y=x). Portanto, quais mais os pontos ficarem ao redor da linha, maior a fidelidade do MDS. Além disso, foi calculada alta correlação de Pearson, próximo a 1 (r=0.9457).

A partir desses resultados, é possivel entender que o MDS Métrico obteve alto desempenho na redução da dimansionalidade dos dados.

6.4 Impressão dos gráficos: 2D PC1xPC2 e 2D MDS (25D)

In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

plot_embedding(axes[0], coords_baseline, y, title=f"Referência Limitada (Apenas PC1 e PC2)\nFidelidade = {corr_baseline:.4f}", show_centroids=True)
plot_embedding(axes[1], coords_best, y, title=f"Mapa MDS Métrico (25D -> 2D)\nFidelidade = {best['corr']:.4f}", show_centroids=True)
plt.tight_layout()
plt.show()


O objetivo deste MVP foi alcançado com sucesso ao utilizar a técnica de MDS Métrico para comprimir as 25 dimensões latentes do PCA (90,0% da variância original) em um espaço 2D interpretável. A melhor solução configurada (n_init=4) provou-se robusta, atingindo uma forte correlação de distâncias de Pearson (r = 0,9457 - 94,57% de fidelidade geométrica), o que representou um ganho real de \(+0,0130\) em relação à referência simplificada do PCA 2D (r = 0,9327). Além disso, o MDS permitiu avaliar que há dispersão entre cada espécie; diferente do que as duas dimensões representadas com PCA mostram.

Para avançar na interpretação biológica do MDS, é necessária a transição da análise macroestrutural para a análise atômica local: "Quais pares de resíduos apresentam o memsmo padrão de correlação entre as espécies? E quais pares apresentam diferente correlação entre SARS-CoV, MERS-CoV e SARS-CoV2?